# 04 Neurosymbolic Inference Heatmap

- The frozen detector still provides the backbone, RPN, proposals, RoI pooling, and neural bbox regression.
- The symbolic classifier replaces only the RoI classification branch.
- Explanations come from the sparse oblique tree path weights reshaped back into the pooled RoI lattice, not from Grad-CAM.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from neuro.datasets import DeepPCBDataset
from neuro.inference import visualize_prediction
from neuro.transforms import build_eval_transforms
from neuro.utils import latest_run_checkpoint, load_yaml
from neurosym.inference import (
    aggregate_detection_heatmaps,
    explain_hybrid_detections,
    load_neurosymbolic_detector,
    run_neurosymbolic_inference,
    select_detection_indices,
    subset_detection,
)

PROJECT_ROOT = Path.cwd().resolve()


In [ ]:
model_config = load_yaml(PROJECT_ROOT / "configs/model.yaml")
train_config = load_yaml(PROJECT_ROOT / "configs/train_deeppcb.yaml")

dataset_root = PROJECT_ROOT / train_config["dataset"]["root"]
checkpoint_path = latest_run_checkpoint(PROJECT_ROOT / train_config["artifacts"]["checkpoint_dir"])
symbolic_checkpoint_path = PROJECT_ROOT / "checkpoints" / "symbolic" / "sodt_run1.pt"
selection_lane = "paper_faithful"

hybrid_model, detector_checkpoint = load_neurosymbolic_detector(
    checkpoint_path,
    PROJECT_ROOT / "configs/model.yaml",
    symbolic_checkpoint_path,
    device=train_config.get("device"),
    selection_lane=selection_lane,
)

test_dataset = DeepPCBDataset(
    dataset_root=dataset_root,
    split_file=train_config["dataset"]["test_split"],
    transforms=build_eval_transforms(),
    class_names=tuple(model_config["model"]["class_names"]),
)

checkpoint_path, symbolic_checkpoint_path, selection_lane


In [ ]:
sample_index = 0
image, target = test_dataset[sample_index]
detection = run_neurosymbolic_inference(hybrid_model, [image])[0]

fig, ax = plt.subplots(figsize=(8, 8))
visualize_prediction(
    image,
    detection,
    tuple(model_config["model"]["class_names"]),
    score_threshold=0.3,
    ax=ax,
)
ax.set_title("Hybrid neurosymbolic prediction")
plt.show()

detection["labels"][:10], detection["scores"][:10]


In [ ]:
score_threshold = 0.3
max_explanations = 9
explanation_mode = "local_instance_evidence_map"
structural_mode = "global_or_structural_density_map"

selected_indices = select_detection_indices(
    detection,
    score_threshold=score_threshold,
    max_detections=max_explanations,
)

if selected_indices:
    selected_detection = subset_detection(detection, selected_indices)
    explanations = explain_hybrid_detections(
        hybrid_model,
        detection,
        image_shape=tuple(image.shape[-2:]),
        detection_indices=selected_indices,
        mode=explanation_mode,
    )
    structural_explanations = explain_hybrid_detections(
        hybrid_model,
        detection,
        image_shape=tuple(image.shape[-2:]),
        detection_indices=selected_indices,
        mode=structural_mode,
    )
    combined_heatmap = aggregate_detection_heatmaps(
        explanations,
        reduction="max",
        score_weighted=True,
        normalize=True,
    )
    combined_structural_heatmap = aggregate_detection_heatmaps(
        structural_explanations,
        reduction="max",
        score_weighted=True,
        normalize=True,
    )

    image_array = image.detach().cpu().permute(1, 2, 0).clamp(0.0, 1.0).numpy()
    class_names = tuple(model_config["model"]["class_names"])

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    visualize_prediction(
        image,
        selected_detection,
        class_names,
        score_threshold=0.0,
        ax=axes[0],
    )
    axes[0].set_title("Selected hybrid detections")
    axes[1].imshow(image_array, cmap="gray" if image_array.shape[-1] == 1 else None)
    axes[1].imshow(combined_heatmap, cmap="inferno", alpha=0.65)
    axes[1].set_title("Combined local instance evidence")
    axes[1].axis("off")
    axes[2].imshow(image_array, cmap="gray" if image_array.shape[-1] == 1 else None)
    axes[2].imshow(combined_structural_heatmap, cmap="magma", alpha=0.65)
    axes[2].set_title("Combined structural density map")
    axes[2].axis("off")
    plt.show()

    fig, axes = plt.subplots(len(explanations), 3, figsize=(16, 4 * len(explanations)))
    if len(explanations) == 1:
        axes = [axes]

    for row, (explanation, structural_explanation) in enumerate(zip(explanations, structural_explanations)):
        explained_detection = subset_detection(detection, [explanation["detection_index"]])
        visualize_prediction(
            image,
            explained_detection,
            class_names,
            score_threshold=0.0,
            ax=axes[row][0],
        )
        label_name = class_names[explanation["label"] - 1]
        axes[row][0].set_title(
            f"Detection {explanation['detection_index']} | {label_name} {explanation['score']:.2f}"
        )
        axes[row][1].imshow(explanation["detection_box_heatmap"], cmap="inferno")
        axes[row][1].set_title("Local instance evidence map")
        axes[row][1].axis("off")
        axes[row][2].imshow(structural_explanation["detection_box_heatmap"], cmap="magma")
        axes[row][2].set_title("Structural density map")
        axes[row][2].axis("off")

    plt.tight_layout()
    plt.show()

    focus_explanation = explanations[0]
    focus_structural_explanation = structural_explanations[0]
    focus_label = class_names[focus_explanation["label"] - 1]

    node_summary_table = pd.DataFrame(
        [
            {
                "node_index": node["node_index"],
                "decision": "left" if node.get("went_left", False) else "right",
                "score": node.get("score"),
                "active_original_features": node["active_original_feature_count"],
                "top_structural_channels": [entry["channel"] for entry in node["top_structural_channels"][:3]],
                "top_positive_local_channels": [entry["channel"] for entry in node.get("top_positive_local_channels", [])[:3]],
                "top_positive_local_coords": [
                    (entry["row"], entry["col"])
                    for entry in node.get("top_positive_local_coordinates", [])[:3]
                ],
            }
            for node in focus_explanation["node_summaries"]
        ]
    )

    top_positive_feature_table = pd.DataFrame(focus_explanation["top_positive_contributing_features"][:10])
    top_negative_feature_table = pd.DataFrame(focus_explanation["top_negative_contributing_features"][:10])
    top_local_cell_table = pd.DataFrame(focus_explanation["top_positive_local_cells"][:10])
    top_structural_cell_table = pd.DataFrame(focus_structural_explanation["top_structural_cells"][:10])
    top_local_channel_table = pd.DataFrame(focus_explanation["top_positive_local_channels"][:8])
    top_structural_channel_table = pd.DataFrame(focus_structural_explanation["top_structural_channels"][:8])

    display(node_summary_table)
    display(top_positive_feature_table)
    display(top_negative_feature_table)
    display(top_local_channel_table)
    display(top_structural_channel_table)
    display(top_local_cell_table)
    display(top_structural_cell_table)

    {
        "focus_detection": {
            "detection_index": focus_explanation["detection_index"],
            "label": focus_label,
            "score": focus_explanation["score"],
            "leaf_index": focus_explanation["leaf_index"],
            "symbolic_label_confidence": focus_explanation["symbolic_label_confidence"],
        },
        "intrinsic_reasoning_summary": focus_explanation["intrinsic_reasoning_summary"],
        "map_types": focus_explanation["reasoning_summary"]["map_types"],
    }
else:
    print("No detections above the current score threshold.")
